# Camera-ready experiments: *Does Weight Decay Enhance Training Stability?* (NeurIPS 2026)

This notebook runs every new experiment for the camera-ready. The code is the `camera-ready` branch of
[lutscha/slt-experiments](https://github.com/lutscha/slt-experiments/tree/camera-ready).

## How to run it
1. **Runtime → Change runtime type → GPU.** Use an A100 or L4 if available; a T4 works but is slower.
2. **Run cells 1–3 (setup).** Cell 1 asks for permission to mount Google Drive, where all results are saved.
3. **Run cell 4 (smoke test, about 5–10 min).** It runs every experiment for a few steps and checks the outputs.
   It must end with **`SMOKE TEST PASSED`**. If it doesn't, stop and send the error output.
4. **Run cell 5 (all experiments) and leave it overnight.** Keep the browser tab open and the computer awake.
   Free Colab can still disconnect after some hours. If that happens, reconnect, run cells 1–3, then cell 5 again:
   finished runs are skipped, so it carries on from the run that was interrupted.
5. **Cell 6 (progress)** can be run at any time, from a second tab, to see which runs have finished.
6. **In the morning, run cell 7 (analysis).** Then send the folder
   `MyDrive/slt_camera_ready/camera_ready_out/` (CSVs and plots). If possible, also send the `log_*.txt` files.

## What runs (in this order)
| Group | Runs | Purpose | Rough time on an L4 |
|---|---|---|---|
| `layernorm` | MLP, full-LayerNorm MLP, hidden-only-LayerNorm MLP at γ=0.05; full-LayerNorm at γ=0, 0.02 (η=0.02, 10k steps) | Reviewer xTi3: LayerNorm traces, c_y^crit ∝ S^d, symmetry decomposition | 2–4 h |
| `ntk` | MLP, η=0.02, 10 values of γ from 1e-5 to 2.5e-2 | Reviewer DAe5: NTK top eigenvalue and KTA across γ | 1–2 h |
| `ntk_convergence` | NTK at n = 100…1000 samples | NTK sample-size check | ~5 min |
| `adamw` | Full-batch AdamW, η=1e-4, γ ∈ {0, 1, 4, 10} | Reviewer DAe5: preconditioned sharpness vs. 38/η − 19γ | 1–2 h |

The times are rough estimates. Cell 4 prints how long each smoke run took on your GPU.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the code

In [ ]:
import os
if not os.path.isdir('/content/slt-experiments'):
    !git clone -b camera-ready https://github.com/lutscha/slt-experiments.git /content/slt-experiments
else:
    !cd /content/slt-experiments && git pull
%cd /content/slt-experiments
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Paths (CIFAR-10 is downloaded automatically on first use)

In [ ]:
import os
os.environ['DATASETS'] = '/content/datasets'
os.environ['RESULTS'] = '/content/drive/MyDrive/slt_camera_ready'
os.makedirs(os.environ['DATASETS'], exist_ok=True)
os.makedirs(os.environ['RESULTS'], exist_ok=True)
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
print('Results will be saved to', os.environ['RESULTS'])

## 4. Smoke test (must print `SMOKE TEST PASSED`)
This runs every group for a few steps on a 1k-sample subset. Results go to `/content/smoke`, not Drive.

In [ ]:
import os, shutil, subprocess, time
env = dict(os.environ, RESULTS='/content/smoke')
shutil.rmtree('/content/smoke', ignore_errors=True)
commands = [
    'camera_ready/run.py layernorm --smoke --only 0',
    'camera_ready/run.py layernorm --smoke --only 1',
    'camera_ready/run.py layernorm --smoke --only 2',
    'camera_ready/run.py ntk --smoke --only 0',
    'camera_ready/run.py ntk_convergence --smoke',
    'camera_ready/run.py adamw --smoke --only 1',
    'camera_ready/analyze.py --smoke',
    'camera_ready/smoke_check.py',
]
for c in commands:
    t = time.time()
    p = subprocess.run('python ' + c, shell=True, env=env, capture_output=True, text=True)
    print(f'[{time.time() - t:6.1f}s] {c}')
    if p.returncode != 0 or c.endswith('smoke_check.py'):
        print(p.stdout[-4000:]); print(p.stderr[-4000:])
    if p.returncode != 0:
        raise RuntimeError(f'Failed: {c}  (send the output above)')

## 5. All experiments (leave running overnight)
Runs the groups one after another. Logs go to `MyDrive/slt_camera_ready/log_<group>.txt`. A group that fails
does not stop the others. Re-running this cell after a disconnect skips finished runs.

In [ ]:
import os, subprocess, time
groups = ['layernorm', 'ntk', 'ntk_convergence', 'adamw']
for g in groups:
    t = time.time()
    log = f"{os.environ['RESULTS']}/log_{g}.txt"
    print(f'=== {g} started {time.strftime("%H:%M:%S")}, log: {log}', flush=True)
    with open(log, 'a') as f:
        p = subprocess.run(['python', 'camera_ready/run.py', g], stdout=f, stderr=subprocess.STDOUT)
    status = 'OK' if p.returncode == 0 else f'FAILED (exit {p.returncode}), see log'
    print(f'=== {g} {status} after {(time.time() - t) / 60:.0f} min', flush=True)
!python camera_ready/analyze.py
!python camera_ready/status.py

## 6. Progress (run any time)

In [ ]:
!python camera_ready/status.py
!tail -n 3 $RESULTS/log_*.txt

## 7. Analysis: CSVs and quick-look plots in `MyDrive/slt_camera_ready/camera_ready_out/`

In [ ]:
!python camera_ready/analyze.py
import glob, os
from IPython.display import Image, display
for f in sorted(glob.glob(os.environ['RESULTS'] + '/camera_ready_out/*.png')):
    print(f); display(Image(f))